# 03_sensitivity
需要・原価・単価シナリオを適用し、配賦結果の粗利インパクトを比較する。

In [1]:
from pathlib import Path
import sys
import pandas as pd

NOTEBOOK_ROOT = Path.cwd()
PROJECT_ROOT = NOTEBOOK_ROOT.parent
sys.path.append(str(PROJECT_ROOT / 'scripts'))

from allocation_utils import CapacityConfig, build_option_table, greedy_allocate, summarize_allocation
from sensitivity_utils import list_default_scenarios, apply_scenario

CAPACITY = {'A': 528000, 'B': 528000}
config = CapacityConfig(plant_capacity=CAPACITY, capacity_utilization_target=0.9)


In [2]:
data_dir = PROJECT_ROOT / 'data' / 'intermediate'
margin_matrix_base = pd.read_parquet(data_dir / 'margin_matrix.parquet')
segment_demand_base = pd.read_parquet(data_dir / 'segment_demand.parquet')
scenarios = list_default_scenarios()
len(scenarios)


7

In [3]:
records = []
scenario_details = {}
for scenario in scenarios:
    margin_matrix, segment_demand = apply_scenario(margin_matrix_base, segment_demand_base, scenario)
    options = build_option_table(margin_matrix)
    allocation_df, plant_remaining, demand_remaining = greedy_allocate(options, segment_demand, config)
    summaries = summarize_allocation(allocation_df, config, segment_demand, plant_remaining, demand_remaining)
    total_margin = allocation_df['alloc_margin'].sum() if not allocation_df.empty else 0
    total_qty = allocation_df['alloc_qty'].sum() if not allocation_df.empty else 0
    records.append({
        'scenario': scenario.name,
        'allocated_qty': total_qty,
        'total_margin': total_margin,
        'avg_unit_margin': total_margin / total_qty if total_qty else 0,
    })
    scenario_details[scenario.name] = {
        'plant': summaries.get('plant'),
        'segment': summaries.get('segment'),
    }
scenario_df = pd.DataFrame(records)
scenario_df


,scenario,allocated_qty,total_margin,avg_unit_margin
0,Base,6732.061111,1.051113e+07,1561.353501
1,DemandPlus10,7247.500556,1.093242e+07,1508.440157
2,DemandMinus10,6058.855000,9.978847e+06,1646.985685
3,CostPlus5,6732.061111,9.821921e+06,1458.976797
4,CostMinus5,6732.061111,1.120033e+07,1663.730205
5,PricePlus5,6732.061111,1.172589e+07,1741.797880
6,PriceMinus5,6732.061111,9.296365e+06,1380.909122


In [4]:
output_dir = PROJECT_ROOT / 'data' / 'intermediate'
scenario_df.to_csv(output_dir / 'scenario_results.csv', index=False)
print('scenario_results.csv を書き出しました。')


scenario_results.csv を書き出しました。
